# Day 5-01｜投籃分析 Pipeline 摘要
> Python 籃球運動資料分析課程  
> 讀取 Day 4 的姿態與球軌跡結果，整理成展示用摘要指標。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 讀取 pose angle 與 ball track 輸出。
- 計算簡單的投籃摘要數值。
- 把數字和 Day 4 的動態 overlay 對起來看，知道每個指標在畫面上代表什麼。

## 完成產出
- `d5_01_shooting_summary.json` 分析摘要檔。

## 課堂要求
- 先完成 Day 4 的姿態與球軌跡分析。
- 按照本單元順序執行各段程式。
- 完成指定輸出後，記錄結果並供課堂討論。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 尋找 Day 4 結果檔。
2. 計算摘要數值。
3. 先看 Day 4 overlay，再把成果寫成 JSON。


In [1]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


In [ ]:
import json
import pandas as pd
from src.shooting_utils import estimate_release_frame

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_03_pose_angles.csv"
ball_csv = RESULTS / "d4_04_ball_track.csv"

if not pose_csv.exists():
    raise FileNotFoundError(f"找不到姿態結果：{pose_csv}。請先完成 Day 4-03。")
if not ball_csv.exists():
    raise FileNotFoundError(f"找不到球軌跡結果：{ball_csv}。請先完成 Day 4-04。")

pose_df = pd.read_csv(pose_csv)
ball_df = pd.read_csv(ball_csv)

if pose_df.empty:
    raise RuntimeError("d4_03_pose_angles.csv 是空的，請重新執行 Day 4-03。")
if ball_df.empty:
    raise RuntimeError("d4_04_ball_track.csv 是空的，請重新執行 Day 4-04。")

pose_df.head()


In [ ]:
summary = {
    "pose_frames": int(len(pose_df)),
    "max_elbow_angle": float(pose_df["elbow_angle"].max()),
    "min_elbow_angle": float(pose_df["elbow_angle"].min()),
    "max_knee_angle": float(pose_df["knee_angle"].max()),
    "ball_points": int(len(ball_df)),
    "estimated_release_frame": estimate_release_frame(ball_df),
}
summary


In [ ]:
from src.video_utils import display_video_in_notebook

print(f"抓到 {summary['pose_frames']} 個姿態 frame、{summary['ball_points']} 個球軌跡點。")
print(f"估計出手 frame: {summary['estimated_release_frame']}")

for preview in [
    RESULTS / "d4_03_pose_overlay_preview.mp4",
    RESULTS / "d4_04_ball_overlay_preview.mp4",
]:
    if preview.exists():
        print("preview:", preview.name)
        display_video_in_notebook(preview, width=720, muted=True, loop=True)


In [ ]:
out_json = RESULTS / "d5_01_shooting_summary.json"
out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", out_json)

這份 summary 不是正式運動科學評估，只是讓學生知道如何把影像分析轉成可報告的數字。建議先看 Day 4 的 overlay，再回頭讀這些數值，會比較知道它們在畫面中對應的是哪一段動作。

## 本單元產出檔案

- `assets/results/d5_01_shooting_summary.json`：投籃分析摘要。
